# Evolução do estoque de veículos — agosto a outubro de 2026

**Objetivo:** comparar duas fotografias de estoque, acompanhar a cobertura de preços e localizar a concentração do valor anunciado nos itens com mais de 180 dias em pátio.

**Entradas:** `estoque_lucas.csv` (03/08/2026) e `estoque_2026-10-15_anonimizado.csv` (15/10/2026).

> O valor usado é a soma dos preços anunciados. Não representa custo de aquisição, investimento contábil ou margem de lucro.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE = Path.cwd()
old = pd.read_csv(BASE / "estoque_lucas.csv", sep=";", encoding="utf-8-sig")
new = pd.read_csv(BASE / "estoque_2026-10-15_anonimizado.csv", sep=";", encoding="utf-8-sig")

# A data exata de entrada não é necessária para comparar a idade informada.
old = old.drop(columns=["entrada"], errors="ignore")

def to_bool(s):
    return s.map(lambda x: str(x).strip().lower() in {"true", "1", "sim", "yes"} if pd.notna(x) else False)

def prepare(df, date):
    d = df.copy()
    d["preco"] = pd.to_numeric(d["preco"], errors="coerce")
    d["dias"] = pd.to_numeric(d["dias"], errors="coerce")
    for col in ["locado", "repasse"]:
        if col in d.columns:
            d[col] = to_bool(d[col])
        else:
            d[col] = False
    if "locadora" in d.columns:
        d["locadora"] = to_bool(d["locadora"])
    else:
        d["locadora"] = d["modelo"].fillna("").astype(str).str.contains(r"\bLOCADORA\b", case=False, regex=True)
    if "indisponivel" in d.columns:
        d["indisponivel"] = to_bool(d["indisponivel"])
    else:
        d["indisponivel"] = d["modelo"].fillna("").astype(str).str.contains("INDISPON[IÍ]VEL", case=False, regex=True)
    # As etiquetas LOCADO/LOCADORA são preservadas como marcadores da fonte.
    # Não equivalem automaticamente à categoria operacional "ex-locado".
    d["categoria"] = "Normal"
    d.loc[d["repasse"], "categoria"] = "Repasse"
    d.loc[d["locado"], "categoria"] = "Marcado LOCADO (status a confirmar)"
    d.loc[d["locadora"], "categoria"] = "Marcado LOCADORA (status a confirmar)"
    d["situacao_locacao"] = "Sem marcador LOCADO/LOCADORA na fonte"
    d.loc[d["locado"], "situacao_locacao"] = "Marcado LOCADO; não classificar automaticamente como ex-locado"
    d.loc[d["locadora"], "situacao_locacao"] = "Marcado LOCADORA; status operacional a confirmar"
    d["data_referencia"] = date
    return d

old = prepare(old, "03/08/2026")
new = prepare(new, "15/10/2026")

assert len(old) == 184
assert len(new) == 163
assert "entrada" not in new.columns
assert new["preco"].dropna().ge(0).all()
assert new["dias"].dropna().ge(0).all()

print(f"Agosto: {len(old)} registros; {old.preco.notna().sum()} com preço.")
print(f"Outubro: {len(new)} registros; {new.preco.notna().sum()} com preço.")
print(f"Indisponíveis: {old.indisponivel.sum()} em agosto; {new.indisponivel.sum()} em outubro.")

## 1. Indicadores principais

A tabela mostra todos os registros do relatório, inclusive os marcados como indisponíveis. A leitura excluindo indisponíveis aparece separadamente.

In [ ]:
def summarize(df):
    priced = df[df["preco"].notna()]
    available = priced[~priced["indisponivel"]]
    old180 = priced[priced["dias"] > 180]
    available180 = available[available["dias"] > 180]
    total = priced["preco"].sum()
    av_total = available["preco"].sum()
    return {
        "Data": df["data_referencia"].iloc[0],
        "Registros": len(df),
        "Carros": int((df["tipo"] == "carro").sum()),
        "Motos": int((df["tipo"] == "moto").sum()),
        "Com preço": len(priced),
        "Sem preço": int(df["preco"].isna().sum()),
        "Cobertura (%)": round(len(priced) / len(df) * 100, 2),
        "Soma anunciada (R$)": int(total),
        "Indisponíveis com preço": int(priced["indisponivel"].sum()),
        ">180 dias com preço": len(old180),
        "Valor >180 dias (R$)": int(old180["preco"].sum()),
        "% valor >180 dias": round(old180["preco"].sum() / total * 100, 2),
        "Disponíveis com preço": len(available),
        "Valor disponível (R$)": int(av_total),
        "Disponíveis >180 dias": len(available180),
        "Valor disponível >180 dias (R$)": int(available180["preco"].sum()),
        "% disponível >180 dias": round(available180["preco"].sum() / av_total * 100, 2) if av_total else 0,
    }

summary = pd.DataFrame([summarize(old), summarize(new)]).set_index("Data")
summary.T

## 2. Distribuição do valor anunciado por idade do estoque

Os valores consideram somente veículos com preço informado. A diferença de cobertura de preços entre as fotografias limita a comparação direta.

In [ ]:
def aging_table(df, label):
    p = df[df["preco"].notna()]
    ranges = [
        ("0–30 dias", (p["dias"] >= 0) & (p["dias"] <= 30)),
        ("31–90 dias", (p["dias"] > 30) & (p["dias"] <= 90)),
        ("91–180 dias", (p["dias"] > 90) & (p["dias"] <= 180)),
        ("Mais de 180 dias", p["dias"] > 180),
    ]
    total = p["preco"].sum()
    return [{"Data": label, "Faixa": name, "Veículos": int(mask.sum()),
             "Valor anunciado (R$)": int(p.loc[mask, "preco"].sum()),
             "% valor": round(p.loc[mask, "preco"].sum() / total * 100, 2) if total else 0}
            for name, mask in ranges]

aging = pd.DataFrame(aging_table(old, "03/08/2026") + aging_table(new, "15/10/2026"))
aging

In [ ]:
pivot = aging.pivot(index="Faixa", columns="Data", values="Valor anunciado (R$)")
pivot = pivot.reindex(["0–30 dias", "31–90 dias", "91–180 dias", "Mais de 180 dias"])
ax = pivot.plot(kind="bar", figsize=(11, 5.5), color=["#7da7c7", "#e87940"])
ax.set_title("Valor anunciado por tempo em estoque", fontsize=14)
ax.set_ylabel("Valor anunciado (R$)")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=0)
ax.grid(axis="y", alpha=0.25)
ax.legend(title="Data do relatório")
plt.tight_layout()
plt.show()

## 3. Dias em estoque por categoria

A média e a mediana descrevem a idade dos veículos que permaneciam no estoque. Não são taxas diretas de giro nem demonstram causalidade.

In [ ]:
def category_table(df, label):
    p = df[df["preco"].notna()].copy()
    result = p.groupby("categoria").agg(
        veiculos=("dias", "size"),
        media_dias=("dias", "mean"),
        mediana_dias=("dias", "median")
    ).reset_index()
    result.insert(0, "Data", label)
    result[["media_dias", "mediana_dias"]] = result[["media_dias", "mediana_dias"]].round(1)
    return result

categories = pd.concat([
    category_table(old, "03/08/2026"),
    category_table(new, "15/10/2026")
], ignore_index=True)
categories.sort_values(["Data", "categoria"])

## 4. Veículos marcados como repasse

Esta seção soma os preços anunciados dos veículos com o marcador `repasse=True`. Não é custo nem margem de lucro. Registros sem preço, caso existam nessa categoria, não entram na soma.

In [ ]:
def repasse_summary(df, label):
    priced = df[df["preco"].notna()]
    rep = priced[priced["repasse"]]
    cars = rep[rep["tipo"] == "carro"]
    motos = rep[rep["tipo"] == "moto"]
    older = rep[rep["dias"] > 180]
    total = priced["preco"].sum()
    return {
        "Data": label,
        "Veículos de repasse com preço": len(rep),
        "Valor anunciado de repasse (R$)": int(rep["preco"].sum()),
        "Carros de repasse": len(cars),
        "Valor anunciado carros (R$)": int(cars["preco"].sum()),
        "Motos de repasse": len(motos),
        "Valor anunciado motos (R$)": int(motos["preco"].sum()),
        "Repasse com mais de 180 dias": len(older),
        "Valor de repasse >180 dias (R$)": int(older["preco"].sum()),
        "Repasse / valor total anunciado (%)": round(rep["preco"].sum() / total * 100, 2) if total else 0,
    }

repasse_comparison = pd.DataFrame([
    repasse_summary(old, "03/08/2026"),
    repasse_summary(new, "15/10/2026"),
])
repasse_comparison.set_index("Data").T

## 5. Alterações de preço e correspondência agregada

A comparação privada entre os relatórios identificou **120 registros correspondentes**. Em 76 pares com preço disponível nas duas datas, seis apresentaram mudança de preço, com variação líquida de **−R$ 58.000**.

Os códigos usados para o pareamento não fazem parte dos arquivos públicos. Por isso, essa estatística é apresentada em nível agregado, sem expor identificadores por unidade. A ausência de um registro num dos relatórios não comprova, isoladamente, venda, entrada ou baixa específica.

## 6. Interpretação e limitações

- O estoque total passou de 184 para 163 registros (redução líquida de 21).
- A cobertura de preço aumentou de 66,85% para 92,02%, portanto os totais anunciados não são perfeitamente comparáveis.
- Em outubro, 38 registros com preço superavam 180 dias, somando R$ 4.531.630 (53,51% do valor anunciado com preço). Dois desses registros estavam marcados como indisponíveis; excluindo-os, são 36 registros e R$ 4.194.650 (51,59% do valor anunciado disponível).
- No relatório de outubro, **37 veículos marcados como repasse** têm preço anunciado, totalizando **R$ 1.699.550**. O indicador é baseado na classificação apresentada no relatório.
- A operação informou que há **3 carros ex-locados disponíveis para venda**, enquanto os demais veículos marcados como LOCADO seguem em locação. Como os três carros ainda não foram vinculados com segurança às linhas públicas anonimizadas, o notebook não calcula sua idade média ou valor anunciado específico.
- As médias por categoria são apenas descritivas; não são taxas de giro nem demonstram causalidade.
- Preço anunciado não equivale a custo, preço efetivo de venda, caixa ou lucro.
- Registros parecidos podem representar unidades distintas e não são removidos automaticamente.
- Os arquivos públicos omitem identificadores sensíveis. O relatório operacional original não faz parte do repositório.